# IE 643: v1 baseline submission (MLP on the 768-d features)
Goal: a solid, fast first `submission.csv`, plus saved OOF/test probabilities so later versions (distillation, pseudo-labeling) can be ensembled with it.

Design choices come from the EDA: ~90% of train *and* test is deformed, class 0 is 35% of the labels (rare classes matter for weighted accuracy), and the deformation is easy to detect (AUC 0.998).


In [ ]:
!pip -q install -U kagglehub
import os, glob, kagglehub
kagglehub.login()
path = kagglehub.competition_download('ie-643-2026-programming-challenge')
print("Path to competition files:", path)

100%|██████████| 11.8M/11.8M [00:00<00:00, 80.1MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/ie-643-2026-programming-challenge
Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

SEED = 0

train  = pd.read_csv(f"{path}/Train_Data.csv")
test   = pd.read_csv(f"{path}/Test_Data.csv")
sample = pd.read_csv(f"{path}/Sample_Submission.csv")

Xtr = train.iloc[:, 1:769].values.astype(np.float64)
Xte = test.iloc[:, 1:769].values.astype(np.float64)
deformed = train["is_deformed"].values.astype(int)
y = train["Class_Label"].values.astype(int)
lab = y != -1
lab_idx = np.where(lab)[0]
print(train.shape, test.shape, "| labeled:", lab.sum(), "| deformed frac:", deformed.mean().round(3))
print("sample submission head:\n", sample.head(3), "\ntest ids == sample ids:", set(test.sample_id) == set(sample.iloc[:, 0]))

(2076, 771) (490, 769) | labeled: 1284 | deformed frac: 0.9
sample submission head:
     ID  Label
0  1.0    7.0
1  2.0    7.0
2  3.0    7.0 
test ids == sample ids: False


## Preprocessing
Two options, compared by CV: **global** z-scoring, or **group** z-scoring (separately for deformed and clean samples, so the deformation's per-feature shift/shrink is removed). Test samples get a deformed/clean flag from a logistic-regression detector (CV AUC was 0.998). Values are clipped to ±5 because the features are very heavy-tailed (row kurtosis ~120).

In [ ]:
Xall = np.vstack([Xtr, Xte])
Zg = StandardScaler().fit_transform(Xall)

# deformation detector -> flag for test samples
det = LogisticRegression(max_iter=3000, C=0.1).fit(Zg[:len(Xtr)], deformed)
flag_te = (det.predict_proba(Zg[len(Xtr):])[:, 1] > 0.5).astype(int)
flag_all = np.r_[deformed, flag_te]
print("test flagged deformed:", flag_te.mean().round(3))

def prep(mode):
    if mode == "global":
        Z = Zg.copy()
    elif mode == "group":
        Z = np.empty_like(Xall)
        for g in (0, 1):
            m = flag_all == g
            Z[m] = (Xall[m] - Xall[m].mean(0)) / (Xall[m].std(0) + 1e-6)
    return np.clip(Z, -5, 5)

test flagged deformed: 0.904


## Model: MLP trainer (PyTorch)
Input noise + input dropout for regularization, label smoothing, class-weighted loss (inverse-sqrt frequency) so rare classes are not ignored.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

class MLP(nn.Module):
    def __init__(self, d_in, n_cls=11, h=512, p=0.5, p_in=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(p_in),
            nn.Linear(d_in, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(p),
            nn.Linear(h, h // 2), nn.BatchNorm1d(h // 2), nn.GELU(), nn.Dropout(p),
            nn.Linear(h // 2, n_cls))
    def forward(self, x): return self.net(x)

def fit_predict(Xa, ya, Xeval_list, seed=0, epochs=80, lr=1e-3, wd=1e-2, noise=0.2, bs=128, ls=0.1):
    """Train on (Xa, ya); return a list of softmax-prob arrays, one per array in Xeval_list."""
    torch.manual_seed(seed); np.random.seed(seed)
    Xa_t = torch.tensor(Xa, dtype=torch.float32, device=dev)
    ya_t = torch.tensor(ya, dtype=torch.long, device=dev)
    cnt = np.bincount(ya, minlength=11).astype(float)
    w = (cnt.sum() / (11 * np.maximum(cnt, 1))) ** 0.5
    w_t = torch.tensor(w / w.mean(), dtype=torch.float32, device=dev)

    model = MLP(Xa.shape[1]).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    n = len(Xa); nb = n // bs                      # drop the last partial batch (BatchNorm-safe)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr * 3, total_steps=epochs * nb)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=dev)
        for b in range(nb):
            idx = perm[b * bs:(b + 1) * bs]
            xb = Xa_t[idx] + noise * torch.randn(len(idx), Xa.shape[1], device=dev)
            loss = F.cross_entropy(model(xb), ya_t[idx], weight=w_t, label_smoothing=ls)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    model.eval(); outs = []
    with torch.no_grad():
        for Xe in Xeval_list:
            outs.append(F.softmax(model(torch.tensor(Xe, dtype=torch.float32, device=dev)), 1).cpu().numpy())
    return outs

device: cpu


## Cross-validation (labeled data only) + test predictions from the fold models
Each fold model also predicts the test set; averaging them gives the submission, so there is no separate final-fit step. Reported metrics are on out-of-fold predictions. **Macro accuracy (mean per-class recall)** is the proxy for the hidden class-weighted metric.

In [ ]:
def metrics(P, name):
    yt = y[lab_idx]; pred = P.argmax(1); ok = pred == yt; d = deformed[lab_idx]
    rec = np.array([ok[yt == c].mean() for c in range(11)])
    print(f"{name:<8} acc={ok.mean():.4f}  macro-acc={rec.mean():.4f}  clean={ok[d==0].mean():.4f}  deformed={ok[d==1].mean():.4f}")
    print("         per-class recall:", np.round(rec, 2).tolist())
    return rec.mean()

def run_cv(Z, n_seeds=2, n_splits=5):
    Ztr, Zte = Z[:len(Xtr)], Z[len(Xtr):]
    oof = np.zeros((len(lab_idx), 11)); tp = np.zeros((len(Zte), 11)); k = 0
    for s in range(n_seeds):
        skf = StratifiedKFold(n_splits, shuffle=True, random_state=s)
        for f, (a, b) in enumerate(skf.split(lab_idx, y[lab_idx])):
            pv, pt = fit_predict(Ztr[lab_idx[a]], y[lab_idx[a]], [Ztr[lab_idx[b]], Zte], seed=1000 * s + f)
            oof[b] += pv / n_seeds; tp += pt; k += 1
    return oof, tp / k

results = {}
for mode in ["global", "group"]:
    oof, tp = run_cv(prep(mode))
    results[mode] = (metrics(oof, mode), oof, tp)

global   acc=0.7897  macro-acc=0.6863  clean=0.9559  deformed=0.7700
         per-class recall: [0.97, 0.7, 0.48, 0.69, 0.74, 0.69, 0.76, 0.82, 0.44, 0.7, 0.55]
group    acc=0.7874  macro-acc=0.6800  clean=0.9485  deformed=0.7683
         per-class recall: [0.97, 0.7, 0.46, 0.7, 0.73, 0.7, 0.78, 0.8, 0.4, 0.69, 0.55]


## Pick the better preprocessing and write `submission.csv`

In [ ]:
best = max(results, key=lambda m: results[m][0])
score, oof_best, tp_best = results[best]
print(f"best preprocessing: {best} (macro-acc {score:.4f})")

# Drop any NaN rows that might be present in the sample submission
sample_clean = sample.dropna(subset=[sample.columns[0]])
sample_clean_ids = set(sample_clean.iloc[:, 0].astype(int))

sub = pd.DataFrame({"ID": test["sample_id"].astype(int), "Label": tp_best.argmax(1).astype(int)})
assert len(sub) == len(sample_clean), f"Length mismatch: test has {len(sub)} rows but sample submission has {len(sample_clean)} valid rows."
assert set(sub.ID) == sample_clean_ids, "ID mismatch with Sample_Submission.csv"

sub.to_csv("submission.csv", index=False)
print(sub.head(), "\npredicted class distribution:\n", sub.Label.value_counts().sort_index().to_string())

# keep probabilities for later ensembling (v2: distillation / pseudo-labeling)
os.makedirs("v1", exist_ok=True)
np.save("v1/oof_probs.npy", oof_best); np.save("v1/test_probs.npy", tp_best)
for m in results: np.save(f"v1/test_probs_{m}.npy", results[m][2])

try:
    from google.colab import files; files.download("submission.csv")
except Exception as e:
    print("(not on Colab; file saved as submission.csv)")

best preprocessing: global (macro-acc 0.6863)
   ID  Label
0   1      5
1   2      8
2   3      3
3   4      7
4   5      3 
predicted class distribution:
 Label
0     73
1     24
2     18
3     49
4     42
5     61
6     59
7     55
8     26
9     55
10    28


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>